# Bronze Ingestion — Customers

Loads the raw `customers.csv` source data from the e-commerce source volume into `ecommerce.bronze.bronze_customers`, keeping raw values untouched. Two metadata columns are added:

* `source_file` — the full path of the source file the row came from
* `ingested_at` — the timestamp when the row was ingested

In [0]:
from pyspark.sql.functions import col, current_timestamp

source_path = "/Volumes/ecommerce/source_data/raw_data/ecomm-raw-data/customers/"
target_table = "ecommerce.bronze.bronze_customers"

# Read the raw CSV as-is (header row present) from the customers directory
df_customers_raw = spark.read.csv(source_path, header=True, inferSchema=True)

# Attach ingestion metadata: full source file path and ingestion timestamp
df_bronze_customers = df_customers_raw.withColumn(
    "source_file", col("_metadata.file_path")
).withColumn(
    "ingested_at", current_timestamp()
)

# Idempotent full load into the bronze table
(
    df_bronze_customers.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.bronze.bronze_customers"))